# Local Voice Translator

Kaggle GPU에서 Gradio 기반 음성 통역 데모를 실행하는 노트북입니다. 번역은 외부 API 없이 `facebook/nllb-200-distilled-600M` 로컬 모델로 처리합니다.

실행 전 오른쪽 Settings에서 Accelerator를 GPU로 변경하세요.

In [ ]:
import os
from pathlib import Path

project_path = Path('/kaggle/working/voice-weight-TTS-STT')
repository_url = 'https://github.com/JooSeunghyeon/voice-weight-TTS-STT.git'

if not project_path.exists():
    !git clone {repository_url} {project_path}

os.chdir(project_path)
print(Path.cwd())

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

In [ ]:
import subprocess
import time
from pathlib import Path

log_path = Path('/kaggle/working/gradio_app.log')
log_file = log_path.open('w')
app_process = subprocess.Popen(
    ['python', 'app.py'],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(15)
print('Gradio app started on http://127.0.0.1:7860')
print('Log file:', log_path)

In [ ]:
import re
import subprocess

tunnel_process = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for line in tunnel_process.stdout:
    print(line, end='')
    match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print('\nPublic Gradio URL:', public_url)
        break

public_url

## 제출 전 체크리스트

- 위 셀의 `Public Gradio URL`에 접속해 Gradio 화면이 열리는지 확인합니다.
- 한국어 음성 -> 외국어, 외국어 음성 -> 한국어를 각각 1회 이상 실행합니다.
- Kaggle notebook 우측 상단 Share에서 권한을 Public으로 변경합니다.
- 실행 화면, 입력/출력 텍스트, 출력 오디오 컴포넌트가 보이도록 스크린샷을 저장합니다.